In [1]:
DATA_DIR = "data/clusterdata-2011-2/"
OUTPUT_DIR = "output/"

In [2]:
import pandas as pd

schema_path = DATA_DIR + "schema.csv"

df = pd.read_csv(schema_path)
df.head()

,file pattern,field number,content,format,mandatory
0,job_events/part-?????-of-?????.csv.gz,1,time,INTEGER,YES
1,job_events/part-?????-of-?????.csv.gz,2,missing info,INTEGER,NO
2,job_events/part-?????-of-?????.csv.gz,3,job ID,INTEGER,YES
3,job_events/part-?????-of-?????.csv.gz,4,event type,INTEGER,YES
4,job_events/part-?????-of-?????.csv.gz,5,user,STRING_HASH,NO


In [3]:
import json

schema_dict = {}
for _, row in df.iterrows():
    file_pattern = row["file pattern"].split("/")[0]
    if file_pattern not in schema_dict:
        schema_dict[file_pattern] = {
            "col_content": [],
            "col_format": [],
            "col_mandatory": []
        }

    schema_dict[file_pattern]["col_content"].append(row["content"])
    schema_dict[file_pattern]["col_format"].append(row["format"])
    schema_dict[file_pattern]["col_mandatory"].append(1 if row["mandatory"] == "YES" else 0)

# Save to JSON
output_file = OUTPUT_DIR + "schema_dict.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(schema_dict, f, ensure_ascii=False, indent=4)

print(f"Schema dictionary has been saved to '{output_file}'")

Schema dictionary has been saved to 'output/schema_dict.json'


## Studying Spark Performance

In [35]:
# Spark default configuration
SPARK_DEFAULTS = {
    "app_name": "GoogleClusterAnalysis",            
    "master": "local[*]",                           
    "spark.sql.shuffle.partitions": 200,          
    "spark.executor.memory": "4g",
    "spark.driver.memory": "4g",
    "spark.arrow.enabled": True,                    
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.executor.cores": 4,
    "spark.sql.autoBroadcastJoinThreshold": 10485760,  # 10 MB
    "cache": False                                 
}

# Config Params Analysis
STRATEGIES = {
    # "spark.sql.shuffle.partitions": [50, 100, 200, 400, 800],
    # "spark.executor.memory": ["2g", "4g", "8g"],
    # "spark.driver.memory": ["2g", "4g"],
    # "spark.executor.cores": [2, 4, 8],
    # "spark.sql.autoBroadcastJoinThreshold": [-1, 10485760, 52428800],
    # "spark.arrow.enabled": [True, False],
    # "spark.sql.adaptive.enabled": [True, False],
    "cache": [True, False]
}


In [36]:
# parse strategies conf to list of structured test conf
# import copy

# cfgs = []

# for key, values in STRATEGIES.items():
#     for v in values:
#         cfg = copy.deepcopy(SPARK_DEFAULTS)

#         # nested key: extra_conf.xxx
#         if key.startswith("extra_conf."):
#             sub_key = key.replace("extra_conf.", "")
#             cfg["extra_conf"][sub_key] = v
#         else:
#             cfg[key] = v

#         cfgs.append(cfg)

# with open(OUTPUT_DIR + "test_conf_list.json", "w") as f:
#     json.dump(cfgs, f, indent=2)

# cfgs

In [37]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, sum as spark_sum, col

def read_csv(spark: SparkSession, name: str):
    return spark.read.option("header", "false").option("inferSchema", "true").option("pathGlobFilter", "*.csv").csv(DATA_DIR + name + "/part-00160-of-00500.csv").toDF(*schema_dict[name]["col_content"])
    
# Create a sample job using 3 tables: task_events, job_events and task_usage 

def run_sample_job(spark: SparkSession, cache: bool = False):
    
    # Read CSVs
    df_task_events, df_job_events, df_task_usage = read_csv(spark, "task_events"), read_csv(spark, "job_events"), read_csv(spark, "task_usage")

    # Join task_events, job_events, task_usage
    df_joint = (
        df_task_events
        .join(df_job_events, on=["job ID", "event type"], how="inner")
        .join(df_task_usage, on=["job ID","task index"], how="inner")
    )

    if cache:
        df_joint = df_joint.cache()

    # Transformation
    # warm up
    df_joint.count()

    # real run
    start_time = time.time()
    (
        df_joint
        .filter(col("event type").isin([1,2,3]))
        .groupBy("job ID")
        .agg(
            count("*").alias("num_tasks"),
            spark_sum("CPU rate").alias("total_cpu"),
            spark_sum("canonical memory usage").alias("total_memory")
        )
        .count()  # trigger action
    )
    dur = time.time() - start_time
    print(f"Job finished in {dur:.2f} seconds")
    return dur


In [38]:
from pyspark.sql import SparkSession
import copy
import time
import pandas as pd

def init_spark(config: dict):
    builder = SparkSession.builder.appName(config.get("app_name", "SparkApp")).master(config.get("master", "local[*]"))

    # Core Spark configs
    for k, v in config.items():
        if k not in ("app_name", "master", "cache"):
            # if v is bool then convert into "true"/"false"
            if isinstance(v, bool):
                v = str(v).lower()
            builder = builder.config(k, v)
            
    # Extra configs
    extra_conf = config.get("extra_conf", {})
    for k, v in extra_conf.items():
        builder = builder.config(k, v)

    # Create session
    spark = builder.getOrCreate()
    print(f"Spark session initialized. Version: {spark.version}")
    
    return spark

def run_experiment(strategies):
    res = []
    for k, vs in strategies.items():
        cfg = copy.deepcopy(SPARK_DEFAULTS)
        for v in vs:
            if k.startswith("extra_conf."):
                sub_key = k.replace("extra_conf.", "")
                cfg["extra_conf"][sub_key] = v
            else:
                cfg[k] = v
                
            spark = init_spark(cfg)
            dur = run_sample_job(spark, cfg["cache"])

            res.append({
                "param": k,
                "value": v,
                "duration": round(dur, 2),
            })

            spark.stop()
            # del spark
            time.sleep(3)

    output_file = OUTPUT_DIR + "experiment_result.csv"
    pd.DataFrame(res).to_csv(output_file, index=False)
    print(f"Experiment finished. Results saved to {output_file}")
            

In [39]:
# # test spark
# from pyspark.sql import SparkSession

# def test():
#     spark = SparkSession.builder \
#         .appName("GoogleClusterAnalysis") \
#         .master("local[*]") \
#         .config("spark.sql.shuffle.partitions", "200") \
#         .config("spark.executor.memory", "4g") \
#         .config("spark.driver.memory", "4g") \
#         .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
#         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
#         .getOrCreate()
    
#     print("Spark version:", spark.version)

#     run_sample_job(spark)

#     spark.stop()
#     del spark

# test()

In [40]:
# Test run experiment
run_experiment(STRATEGIES)

Spark session initialized. Version: 4.1.0


26/01/09 03:19:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/01/09 03:19:25 WARN MemoryStore: Not enough space to cache rdd_49_26 in memory! (computed 43.8 MiB so far)
26/01/09 03:19:25 WARN MemoryStore: Not enough space to cache rdd_49_27 in memory! (computed 43.7 MiB so far)
26/01/09 03:19:25 WARN BlockManager: Persisting block rdd_49_26 to disk instead.
26/01/09 03:19:25 WARN BlockManager: Persisting block rdd_49_27 to disk instead.
26/01/09 03:19:25 WARN MemoryStore: Not enough space to cache rdd_49_23 in memory! (computed 43.8 MiB so far)
26/01/09 03:19:25 WARN BlockManager: Persisting block rdd_49_23 to disk instead.
26/01/09 03:19:27 WARN MemoryStore: Not enough space to cache rdd_49_26 in memory! (computed 22.6 MiB so far)
26/01/09 03:19:27 WARN MemoryStore: Not enough space to cache rdd_49_32 in memory! (computed 22.6 MiB so far)
26/01/09 03:19:27 

Py4JError: An error occurred while calling o3132.count

26/01/09 03:19:36 WARN MemoryStore: Not enough space to cache rdd_49_50 in memory! (computed 43.8 MiB so far)
26/01/09 03:19:37 WARN MemoryStore: Not enough space to cache rdd_49_56 in memory! (computed 22.6 MiB so far)
26/01/09 03:19:37 WARN MemoryStore: Not enough space to cache rdd_49_54 in memory! (computed 43.8 MiB so far)
26/01/09 03:19:37 WARN MemoryStore: Not enough space to cache rdd_49_55 in memory! (computed 22.5 MiB so far)
26/01/09 03:19:37 WARN MemoryStore: Not enough space to cache rdd_49_53 in memory! (computed 43.8 MiB so far)
26/01/09 03:19:38 WARN MemoryStore: Not enough space to cache rdd_49_59 in memory! (computed 43.7 MiB so far)
26/01/09 03:19:38 WARN MemoryStore: Not enough space to cache rdd_49_62 in memory! (computed 22.6 MiB so far)
26/01/09 03:19:38 WARN MemoryStore: Not enough space to cache rdd_49_63 in memory! (computed 43.7 MiB so far)
26/01/09 03:19:38 WARN MemoryStore: Not enough space to cache rdd_49_61 in memory! (computed 43.8 MiB so far)
26/01/09 0